# 실습: 음성 신호를 AM 반송파에 실어 송신하기

지난 실습에서는 **100 Hz 사인파**(규칙적인 신호파)를 반송파에 실어 보냈습니다.
이번에는 **실제 사람 목소리**(마이크 녹음 또는 통화 녹음 파일)를 신호파로 사용합니다.

```
[음성 입력] → [신호파 만들기] → [AM 변조] → [송신(채널+잡음)] → [수신·복조] → [음성 복원]
 녹음/파일     대역제한·정규화   (1+m·x)cosωct   스피커/WAV 저장     포락선 검파
```

| 구분 | 지난 실습 | 이번 실습 |
|---|---|---|
| 신호파 | 100 Hz 사인파 (단일 주파수) | 음성 (300 ~ 3,400 Hz 대역) |
| 스펙트럼 | 반송파 양옆에 선 2개 | 반송파 양옆에 **측파대(대역)** 2개 |

> 실행 순서: 위에서부터 셀을 차례로 실행하세요. (Colab 상단 메뉴 **런타임 → 모두 실행** 가능, 단 녹음 셀에서 마이크 권한 허용 필요)

## 0. 준비 — 라이브러리, 한글 폰트, 실습 파라미터

In [ ]:
import numpy as np, matplotlib.pyplot as plt, matplotlib.font_manager as fm
import subprocess, base64, os
from scipy import signal
from scipy.io import wavfile
from IPython.display import Audio, display, Javascript
from google.colab import output, files

# 한글 폰트 (그래프 제목용)
subprocess.run('apt-get -qq install -y fonts-nanum > /dev/null 2>&1', shell=True)
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rc('font', family='NanumGothic'); plt.rc('axes', unicode_minus=False)
plt.rc('axes', grid=True); plt.rc('grid', color='0.88')

# ===== 실습 파라미터 (바꿔 가며 실험해 보세요) =====
FS     = 48000   # 샘플링 주파수 [Hz]
FC     = 10000   # 반송파 주파수 [Hz]  ※ 스피커로 들을 수 있도록 가청 대역 사용 (실제 AM 방송은 수백 kHz~1.6 MHz)
F_LOW  = 300     # 음성 대역 하한 [Hz]
F_HIGH = 3400    # 음성 대역 상한 [Hz] (전화 품질)
M      = 0.8     # 변조지수 m (0 < m ≤ 1 이면 정상 변조)
SNR_DB = 30      # 채널 잡음 (신호대잡음비, dB) — 작게 하면 잡음 증가
print('준비 완료')

## 1. 음성 입력 — 마이크 녹음 또는 파일 업로드

- `SOURCE = 'record'` : 브라우저 마이크로 바로 녹음 (실행하면 마이크 권한 요청이 뜹니다)
- `SOURCE = 'upload'` : 통화 녹음·음성 파일 업로드 (m4a, mp3, wav, amr, 3gp 등 대부분 지원)

In [ ]:
SOURCE    = 'record'   # 'record' 또는 'upload'
REC_SEC   = 5          # 녹음 시간 [초]
START_SEC = 0          # (파일) 몇 초 지점부터 사용할지
MAX_SEC   = 10         # (파일) 최대 몇 초까지 사용할지 — 긴 통화 녹음 대비

RECORD_JS = '''
const sleep = t => new Promise(r => setTimeout(r, t));
const toDataURL = blob => new Promise(res => {
  const rd = new FileReader(); rd.onloadend = () => res(rd.result); rd.readAsDataURL(blob); });
var record = async (sec) => {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const rec = new MediaRecorder(stream); const chunks = [];
  rec.ondataavailable = e => chunks.push(e.data);
  const msg = document.createElement('p'); document.body.appendChild(msg);
  rec.start();
  for (let i = sec; i > 0; i--) { msg.innerText = '● 녹음 중... ' + i + '초 남음'; await sleep(1000); }
  const stopped = new Promise(r => rec.onstop = r); rec.stop(); await stopped;
  stream.getTracks().forEach(t => t.stop());
  msg.innerText = '녹음 완료';
  return await toDataURL(new Blob(chunks));
}
'''

def record_voice(sec, fname='voice_raw.webm'):
    display(Javascript(RECORD_JS))
    data = output.eval_js(f'record({sec})')
    with open(fname, 'wb') as f:
        f.write(base64.b64decode(data.split(',')[1]))
    return fname

def load_audio(path, fs=FS, start=0, dur=None):
    """ffmpeg로 어떤 형식이든 모노·FS Hz WAV로 변환 후 -1~1 실수 배열로 읽기"""
    cmd = ['ffmpeg', '-y', '-loglevel', 'error', '-ss', str(start), '-i', path]
    if dur: cmd += ['-t', str(dur)]
    cmd += ['-ac', '1', '-ar', str(fs), 'voice.wav']
    subprocess.run(cmd, check=True)
    _, x = wavfile.read('voice.wav')
    return x.astype(np.float32) / 32768.0

if SOURCE == 'record':
    raw = record_voice(REC_SEC)
    x = load_audio(raw)
else:
    up = files.upload()
    raw = list(up.keys())[0]
    x = load_audio(raw, start=START_SEC, dur=MAX_SEC)

print(f'입력: {raw}  |  길이 {len(x)/FS:.2f}초  |  샘플 {len(x):,}개')
display(Audio(x, rate=FS))

## 2. 신호파 만들기 — 대역 제한 + 정규화

음성에는 여러 주파수가 섞여 있습니다. 전화처럼 **300 ~ 3,400 Hz**만 남기고(대역통과 필터),
최대 크기가 1이 되도록 **정규화**합니다. 이것이 변조에 쓸 신호파 $x(t)$ 입니다.

In [ ]:
sos_bp = signal.butter(6, [F_LOW, F_HIGH], btype='band', fs=FS, output='sos')
msg = signal.sosfiltfilt(sos_bp, x)
msg = msg / (np.max(np.abs(msg)) + 1e-12)       # -1 ~ 1 정규화
t = np.arange(len(msg)) / FS

def spectrum_db(sig, fs=FS):
    f, P = signal.welch(sig, fs, nperseg=4096)
    return f, 10*np.log10(P + 1e-15)

# 비교용: 지난 실습의 100 Hz 사인파
sine = np.sin(2*np.pi*100*t)

fig, ax = plt.subplots(3, 1, figsize=(11, 9))
ax[0].plot(t, msg, 'k', lw=0.5); ax[0].set_title('신호파 x(t) — 전체 파형'); ax[0].set_xlabel('시간 [s]')
i0 = np.argmax(np.abs(msg)); w = int(0.03*FS); s0 = max(0, i0-w//2)
ax[1].plot(t[s0:s0+w]*1000, msg[s0:s0+w], 'k', lw=1, label='음성')
ax[1].plot(t[s0:s0+w]*1000, sine[s0:s0+w], color='0.6', ls='--', lw=1, label='100 Hz 사인파(지난 실습)')
ax[1].set_title('30 ms 확대 — 음성은 불규칙한 파형'); ax[1].set_xlabel('시간 [ms]'); ax[1].legend(loc='upper right')
f, Pm = spectrum_db(msg); _, Ps = spectrum_db(sine)
ax[2].plot(f, Pm, 'k', lw=1, label='음성'); ax[2].plot(f, Ps, color='0.6', ls='--', lw=1, label='100 Hz 사인파')
ax[2].set_xlim(0, 5000); ax[2].set_ylim(Pm.max()-80, Pm.max()+10)
ax[2].set_title('스펙트럼 — 사인파는 한 점, 음성은 대역'); ax[2].set_xlabel('주파수 [Hz]'); ax[2].set_ylabel('dB'); ax[2].legend()
plt.tight_layout(); plt.show()
display(Audio(msg, rate=FS))

## 3. AM 변조 — 신호파를 반송파에 싣기

$$s(t) = \big[\,1 + m\,x(t)\,\big]\cos(2\pi f_c t)$$

- 반송파 $\cos(2\pi f_c t)$의 **진폭(포락선)** 이 음성 모양대로 변합니다.
- 스펙트럼에서는 $f_c$를 중심으로 **하측파대(LSB)** 와 **상측파대(USB)** 가 생기고, 점유 대역폭은 $2 \times 3{,}400 = 6{,}800$ Hz 입니다.

In [ ]:
carrier  = np.cos(2*np.pi*FC*t)
envelope = 1 + M*msg
am = envelope * carrier
am_tx = am / np.max(np.abs(am))                 # 송신용 정규화

fig, ax = plt.subplots(3, 1, figsize=(11, 9))
w = int(0.02*FS)
ax[0].plot(t[s0:s0+w]*1000, carrier[s0:s0+w], 'k', lw=0.6)
ax[0].set_title(f'반송파 (f_c = {FC:,} Hz) — 20 ms 확대'); ax[0].set_xlabel('시간 [ms]')
ax[1].plot(t[s0:s0+w]*1000, am[s0:s0+w], color='0.55', lw=0.5, label='AM 피변조파')
ax[1].plot(t[s0:s0+w]*1000,  envelope[s0:s0+w], 'k', lw=1.4, label='포락선 = 1 + m·x(t)')
ax[1].plot(t[s0:s0+w]*1000, -envelope[s0:s0+w], 'k', lw=1.4)
ax[1].set_title(f'AM 피변조파 (m = {M}) — 포락선이 음성 모양'); ax[1].set_xlabel('시간 [ms]'); ax[1].legend(loc='upper right')
f, Pa = spectrum_db(am)
ax[2].plot(f, Pa, 'k', lw=1)
for fx, lab in [(FC-F_HIGH, 'LSB'), (FC, '반송파'), (FC+F_HIGH, 'USB')]:
    ax[2].axvline(fx, color='0.6', ls=':', lw=1)
ax[2].text(FC-F_HIGH/2, Pa.max()-5, 'LSB', ha='center'); ax[2].text(FC+F_HIGH/2, Pa.max()-5, 'USB', ha='center')
ax[2].text(FC, Pa.max()+3, '반송파', ha='center')
ax[2].set_xlim(FC-6000, FC+6000); ax[2].set_ylim(Pa.max()-80, Pa.max()+10)
ax[2].set_title(f'AM 스펙트럼 — 점유 대역폭 {FC-F_HIGH:,} ~ {FC+F_HIGH:,} Hz'); ax[2].set_xlabel('주파수 [Hz]'); ax[2].set_ylabel('dB')
plt.tight_layout(); plt.show()

## 4. 송신 — 스피커 재생 · WAV 저장 · 채널 잡음

- 아래 재생 버튼을 누르면 **AM 신호가 스피커로 송신**됩니다. (10 kHz 반송파의 '삐–' 소리 속에 음성이 섞여 들립니다)
- `am_tx.wav` 로 저장되어 다운로드할 수 있습니다.
- 수신 측에 도달하는 신호는 채널을 거치며 **잡음(AWGN)** 이 더해진다고 가정합니다.

In [ ]:
wavfile.write('am_tx.wav', FS, (am_tx * 32767 * 0.9).astype(np.int16))
print('▶ 송신 신호 (AM)'); display(Audio(am_tx, rate=FS))

# 채널: 백색 가우시안 잡음 추가
p_sig = np.mean(am_tx**2)
noise = np.random.randn(len(am_tx)) * np.sqrt(p_sig / 10**(SNR_DB/10))
rx = am_tx + noise
print(f'채널 SNR = {SNR_DB} dB 적용 → 수신 신호 rx 생성')

# files.download('am_tx.wav')   # 내 PC로 받으려면 주석 해제

## 5. 수신·복조 — 포락선 검파로 음성 복원

AM은 포락선에 정보가 있으므로 **정류(절댓값) → 저역통과필터 → 직류 제거** 만으로 음성을 꺼낼 수 있습니다.
(실제 광석 라디오의 다이오드 + 콘덴서 회로와 같은 원리)

> 💡 확장 실험: 다른 휴대폰으로 4단계의 송신 소리를 **녹음**해서 `RX_FROM_FILE = True` 로 업로드하면, 실제 공기 중(음향 채널)을 거친 신호를 복조해 볼 수 있습니다.

In [ ]:
RX_FROM_FILE = False      # True: 녹음해 온 수신 파일 업로드해서 복조

if RX_FROM_FILE:
    up = files.upload(); rx = load_audio(list(up.keys())[0])

def envelope_detect(sig, fs=FS, cutoff=F_HIGH):
    rect = np.abs(sig)                                         # ① 정류
    sos_lp = signal.butter(6, cutoff, btype='low', fs=fs, output='sos')
    env = signal.sosfiltfilt(sos_lp, rect)                     # ② 저역통과 (포락선)
    out = env - np.mean(env)                                   # ③ 직류(반송파 성분) 제거
    return out / (np.max(np.abs(out)) + 1e-12), rect, env

recovered, rect, env = envelope_detect(rx)

n = min(len(msg), len(recovered))
fig, ax = plt.subplots(2, 1, figsize=(11, 6.5))
w = int(0.02*FS)
ax[0].plot(t[s0:s0+w]*1000, rect[s0:s0+w], color='0.7', lw=0.5, label='① 정류 |rx|')
ax[0].plot(t[s0:s0+w]*1000, env[s0:s0+w], 'k', lw=1.4, label='② 저역통과 후 포락선')
ax[0].set_title('포락선 검파 과정 (20 ms 확대)'); ax[0].set_xlabel('시간 [ms]'); ax[0].legend(loc='upper right')
ax[1].plot(t[:n], msg[:n], color='0.65', lw=0.6, label='원래 신호파')
ax[1].plot(t[:n], recovered[:n], 'k', lw=0.4, ls='--', label='복원된 음성')
ax[1].set_title('원래 신호 vs 복원 신호'); ax[1].set_xlabel('시간 [s]'); ax[1].legend(loc='upper right')
plt.tight_layout(); plt.show()

if not RX_FROM_FILE:
    corr = np.corrcoef(msg[:n], recovered[:n])[0, 1]
    print(f'원래 신호와 복원 신호의 상관계수: {corr:.3f}  (1에 가까울수록 잘 복원)')
print('▶ 복원된 음성'); display(Audio(recovered, rate=FS))
wavfile.write('recovered.wav', FS, (recovered * 32767 * 0.9).astype(np.int16))

## 6. 실험 — 변조지수 m 바꾸기 (과변조)

$m > 1$ 이면 $1 + m\,x(t)$ 가 음수가 되는 구간이 생겨 포락선이 뒤집힙니다(**과변조**).
포락선 검파로 복원하면 음성이 **찌그러지는 것**을 직접 들어 보세요.

In [ ]:
m_list = [0.5, 1.0, 1.5]
seg = slice(s0, s0 + int(0.02*FS))
fig, ax = plt.subplots(len(m_list), 1, figsize=(11, 8), sharex=True)
for a, m in zip(ax, m_list):
    s = (1 + m*msg) * carrier
    a.plot(t[seg]*1000, s[seg], color='0.6', lw=0.5)
    a.plot(t[seg]*1000, np.abs(1 + m*msg[seg]), 'k', lw=1.3)
    a.axhline(0, color='k', lw=0.6)
    a.set_title(f'm = {m}' + ('  ← 과변조: 포락선 왜곡' if m > 1 else ''))
ax[-1].set_xlabel('시간 [ms]'); plt.tight_layout(); plt.show()

for m in m_list:
    s = (1 + m*msg) * carrier
    rec_m, _, _ = envelope_detect(s / np.max(np.abs(s)))
    print(f'▶ m = {m}  복원 음성  (상관계수 {np.corrcoef(msg, rec_m)[0,1]:.3f})')
    display(Audio(rec_m, rate=FS))

## 7. 생각해 볼 문제

1. `SNR_DB` 를 30 → 10 → 0 으로 줄이면 복원 음성은 어떻게 변하는가? AM이 잡음에 약한 이유는?
2. `FC` 를 5000 Hz로 낮추면 스펙트럼에서 무슨 문제가 생기는가? (힌트: $f_c$ 와 음성 대역폭의 관계)
3. 지난 실습(100 Hz 사인파)과 비교해 AM 신호의 **대역폭**은 어떻게 달라졌는가?
4. 실제 AM 라디오 방송국 간격이 9 kHz(한국)인 이유를 음성 대역폭과 연결해 설명해 보자.